In [19]:
import os
import numpy as np
import networkx as nx
import yaml

from scipy.stats import entropy
import math
import pandas as pd

from tqdm import tqdm, trange


In [20]:
algorithm = 'Infomap'
mi = 4

mu =  mi/10 
parts_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/runs'
graphs_folder = f'LFR_Graph_Data/mu_0_{mi}/'
experiment_folder = 'LFR_Graph_Data/Community_Data/Infomap/'

In [21]:
# def calc_C(C, partitions, n_nodes):
#     for i in trange(partitions.shape[0]):
#         part = partitions[i, :]
#         for node1, node2 in itertools.combinations(np.arange(n_nodes), r=2):
#             if part[node1] == part[node2]:
#                 C[node1,node2] += 1
#                 C[node2,node1] += 1
#     C /= partitions.shape[0]
#     np.fill_diagonal(C,1)
#     return C


# Vectorized co-association calculation
def calc_C(C,partitions, n_nodes):
    n_nodes = partitions.shape[1]
    C = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    for i in range(partitions.shape[0]):
        # Vectorized comparison
        part = partitions[i, :]
        same_community = (part[:, None] == part[None, :])
        C += same_community
    C /= partitions.shape[0]
    return C

In [22]:


parts_files = [x for x in os.listdir(parts_folder) if x.endswith('.npy')]
for fil in tqdm(parts_files):
    if f'mu_0_{mi}' in fil:
        print(f"processing {fil}")
        fil_path = os.path.join(parts_folder, fil)
        parts = np.load(fil_path)
        n_nodes = parts.shape[1]
        C = np.zeros((n_nodes, n_nodes))
        C = calc_C(C, parts, n_nodes)
        coassociation_fil = fil.strip('runs.npy') + 'coassociation.npy'
        coassociation_folder = parts_folder.strip('/').strip('Runs') + 'Coassociation'
        if not os.path.exists(coassociation_folder):
            os.mkdir(coassociation_folder)
        coassociation_file = os.path.join(coassociation_folder, coassociation_fil)
        np.save(coassociation_file, C)

  0%|          | 0/360 [00:00<?, ?it/s]

processing graph_01_mu_0_4_runs.npy


 67%|██████▋   | 241/360 [00:02<00:01, 115.86it/s]

processing graph_010_mu_0_4_runs.npy
processing graph_0100_mu_0_4_runs.npy


 67%|██████▋   | 242/360 [00:04<00:02, 51.50it/s] 


KeyboardInterrupt: 

In [ ]:
def calc_node_metrics(G):
    node_degrees = dict(G.degree())
    node_clustering_coefficients = nx.clustering(G)
    node_betweenness = nx.betweenness_centrality(G)
    node_closeness = nx.closeness_centrality(G)
    node_eigenvector = nx.eigenvector_centrality(G, max_iter=1000)
    node_av_shortest_paths = {}
    for i in range(G.number_of_nodes()):
        shortest_paths = nx.algorithms.shortest_paths.generic.shortest_path_length(G, source=i)
        if list(shortest_paths.values())[1:] != []:
            average_shortest_path = np.mean(list(shortest_paths.values())[1:])
        else:
            average_shortest_path = 0
        node_av_shortest_paths[i] = average_shortest_path
    node_metrics = {'Degree': node_degrees, 'Clustering Coefficient': node_clustering_coefficients, 'Betweenness': node_betweenness,
                    'Closeness': node_closeness, 'Shortest Path': node_av_shortest_paths, 'Eigenvector': node_eigenvector}
    return node_metrics, node_degrees


def convert_parts_format(parts):
    final_parts_list = []
    for i in range(parts.shape[0]):
        current_part = parts[i, :]
        converted_part = [[] for _ in range(max(current_part))]
        for node, comm in enumerate(current_part):
            converted_part[comm - 1].append(node) # Subtract 1 since communities are indexed from 1
        final_parts_list.append(converted_part)
    return final_parts_list


def initialise_new_metrics(n_nodes):
    e_in_list = {i: [] for i in range(n_nodes)}
    e_out_list = {i: [] for i in range(n_nodes)}

    e_in_over_e_out = {i: [] for i in range(n_nodes)}
    odf = {i: [] for i in range(n_nodes)}

    expansion = {i: [] for i in range(n_nodes)}
    cut_ratio = {i: [] for i in range(n_nodes)}
    conductance = {i: [] for i in range(n_nodes)}
    normalised_cut = {i: [] for i in range(n_nodes)}

    triangle_participation = {i: [] for i in range(n_nodes)}

    new_metric_dict = {'E In': e_in_list, 'E Out': e_out_list, 'E In Over E Out': e_in_over_e_out,
                       'ODF': odf, 'Expansion': expansion, 'Cut Ratio': cut_ratio,
                       'Conductance': conductance, 'Normalised Cut': normalised_cut,
                       'Triangle Participation': triangle_participation}
    return new_metric_dict


def calc_new_metrics(new_metrics, G, partitions, node_degrees):
    for part in tqdm(partitions):
        for comm in part:

            comm_subgraph = G.subgraph(comm)
            comm_degrees = comm_subgraph.degree()

            w = len(comm)
            N = G.number_of_nodes()
            m = G.number_of_edges()

            for nod in dict(comm_degrees).keys():
                e_in = comm_degrees[nod]
                e_out = node_degrees[nod] - e_in

                new_metrics['E In'][nod].append(e_in)
                new_metrics['E Out'][nod].append(e_out)

                # For e_in divided by e_out, if e_out is 0, just return the value of e_in
                try:
                    new_metrics['E In Over E Out'][nod].append(e_in/e_out)
                except ZeroDivisionError:
                    new_metrics['E In Over E Out'][nod].append(e_in)

                new_metrics['ODF'][nod].append(e_out/node_degrees[nod])

                new_metrics['Expansion'][nod].append(e_out/w)
                try:
                    new_metrics['Cut Ratio'][nod].append(e_out/(N-w))
                except ZeroDivisionError:
                    new_metrics['Cut Ratio'][nod].append(0)

                ct = e_out/(node_degrees[nod] + e_in)
                new_metrics['Conductance'][nod].append(ct)

                nc = ct + e_out/(2*m - 2*e_in + e_out)
                new_metrics['Normalised Cut'][nod].append(nc)

                tp_nods = []
                neighbours = list(comm_subgraph.neighbors(nod))
                for nbr_nod in neighbours:
                    if nbr_nod not in tp_nods:
                        for nbr_nod_2 in neighbours:
                            if comm_subgraph.has_edge(nbr_nod, nbr_nod_2):
                                tp_nods.extend((nbr_nod, nbr_nod_2))
                                break
                tp = len(list(set(tp_nods)))/w
                new_metrics['Triangle Participation'][nod].append(tp)

    return new_metrics


def average_metrics(new_metrics):
    averaged_metrics = new_metrics.copy()
    for met in averaged_metrics.keys():
        for nod in averaged_metrics[met].keys():
            averaged_metrics[met][nod] = np.mean(new_metrics[met][nod])
    return averaged_metrics


def new_node_metrics(node_metrics, partitions, node_degrees, n_nodes):
    new_metrics = initialise_new_metrics(n_nodes)
    new_metrics = calc_new_metrics(new_metrics, G, partitions, node_degrees)
    new_metrics = average_metrics(new_metrics)
    updated_node_metrics = node_metrics.copy()
    updated_node_metrics.update(new_metrics)
    return updated_node_metrics


def append_to_dataframe(X, node_metrics, n_nodes, graph_yml):
    df = pd.DataFrame(node_metrics)
    new_indices = [graph_yml.split('.')[0] + '_node_{0}'.format(k) for k in range(n_nodes)]
    df.index = new_indices
    X = pd.concat([X, df])
    return X


def element_entropy(C):
    E = np.empty_like(C)
    rows, cols = E.shape
    for row in range(rows):
        for col in range(cols):
            p = C[row,col]
            if p > 0:
                E[row,col] = -p * math.log(p, 2)
            else:
                E[row,col] = 0
    entrop = np.mean(E, axis=1)
    return entrop

In [ ]:
if not os.path.exists(os.path.join(experiment_folder, 'Node_Features')):
    os.mkdir(os.path.join(experiment_folder, 'Node_Features'))

if not os.path.exists(os.path.join(experiment_folder, 'Node_Entropies')):
    os.mkdir(os.path.join(experiment_folder, 'Node_Entropies'))

for graph_loc in os.listdir(graphs_folder):
    
    graph_loc = os.path.join(graphs_folder, graph_loc)
    graph_contents = os.listdir(graph_loc)
    graph_yml = [x for x in graph_contents if x.endswith('yml')][0]

    features_path = os.path.join(experiment_folder, 'Node_Features', graph_yml.split('.')[0] + '_features.csv')

    if os.path.exists(features_path):
        print(f'skipping {features_path}')
    else:
        with open(os.path.join(graph_loc, graph_yml)) as f:
            graph_info = yaml.load(f, Loader=yaml.Loader)
        G = graph_info['G']
        n_nodes = graph_info['n']

        features = pd.DataFrame()
        node_entropies = []

        graph_npy = graph_yml.split('.')[0] + '_runs.npy'
        parts_file = os.path.join(experiment_folder, 'runs', graph_npy)
        parts = np.load(parts_file)

        coassociation_npy = graph_yml.split('.')[0] + '_coassociation.npy'
        coassociation_file = os.path.join(experiment_folder, 'rCoassociation', coassociation_npy)
        C = np.load(coassociation_file)

        parts = convert_parts_format(parts)
        node_metrics, node_degrees = calc_node_metrics(G)
        node_metrics = new_node_metrics(node_metrics, parts, node_degrees, n_nodes)

        features = append_to_dataframe(features, node_metrics, n_nodes, graph_yml)
    
        features.to_csv(features_path)

        entropies = element_entropy(C)
        entropies = np.array(entropies)
        node_entropies = pd.DataFrame(entropies, index=features.index, columns=['Entropy'])
        entropies_path = os.path.join(experiment_folder, 'Node_Entropies', graph_yml.split('.')[0] + '_entropies.csv')
        node_entropies.to_csv(entropies_path) 

100%|██████████| 1000/1000 [02:36<00:00,  6.38it/s]
